## Research Outline

#### How to find momentum:
1. Time Horizon
a. Longer time horizons generally lead to momentum.
b. Test different time horizons and see where momentum might exist.

2. New Information / Activity
a. Times of heightened activity coupled with new information favor momentum.
b. Apply indicators of activity / new info (IE. Twitter activity, trading volume) to find stronger momentum.

3. Seasonality
a. Similar seasons tend to show momentum.
b. What are the relevant seasons in crypto and do they show momentum? As we discussed, it could be worthwhile to explore times when institutions vs. retail trade (IE weekdays vs. weekends, day vs. night).

4. Investment Themes
a. Investment “themes” or “styles” tend to show momentum.
b. We discussed some relevant themes in crypto. Do these or other themes you can think of show momentum?

5. Technical Plays
a. Sometimes, predictable mechanical rebalancing leads to momentum.
b. Given the institutional crypto players, and their trading schedules (IE. usually during work-hours), is it possible to front-run them?
 
#### How to find reversal:
1. Time Horizon
a. Shorter time horizons generally lead to reversal.
b. Test different time horizons and see where reversal might exist.

2. Uninformed Trading
a. Uninformed (liquidity-driven) trades reverse more
b. Apply indicators of activity / new info (IE. Twitter activity, trading volume) and isolate cases of lower activity / info to strengthen reversal
c. Potentially draw upon ideas mentioned in the “Fire Sale” crypto video, which relied on uninformed trading during liquidations to find reversal.

3. Correlation
a. Security A – (Something Correlated to It) is more mean-reverting
b. Find either correlated pairs or baskets of crypto assets as discussed in the video on reversal/correlation.

4. Macro
a. There is more reversal in times when there’s more volatility and dislocation in the macro environment.
b. Test if this is true using the indicators of volatility / dislocation we discussed in class (implied volatility, realized volatility, return dispersion, pairwise correlation).

#### Data Sourcing

#### Unconstrained Backtesting

#### Execution / Slippage
Cryptocurrencies can have commissions of ~7bps. While total slippage is unknown and will depend on the trader’s volume as well, let’s assume another 13 bps. So total all-in execution costs will be 20 bps for market-orders. Limit orders will just have the 7 bps of commissions.

#### Weighting

#### Performance Evaluation
 
Please provide the key performance evaluation metrics we’ve discussed in class so far. This includes returns, volatility, sharpes, max drawdowns and alpha / beta.

## Research Question

#### Which entry signal better identifies profitable mean-reversion opportunities in cryptocurrency spread trading?

Baseline: Z-score mean-reversion pairs trading.

Alternative: Bollinger Bands + RSI on the same spread.

Extension: Use a machine learning model (e.g., logistic regression, random forest, or a small neural network) to predict whether a z-score signal is likely to revert successfully.

Individual ranges are unstable -- comparing spreads helps remove the market-wide movement

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import ccxt

In [2]:
# 1) 
# load price data
# align timestamps
# clean missing values
# compute log prices
# optionally resample to a fixed interval

In [3]:
exchange = ccxt.kraken()

eth_raw = exchange.fetch_ohlcv("ETH/USD", timeframe="1h", limit=1000)
sol_raw = exchange.fetch_ohlcv("SOL/USD", timeframe="1h", limit=1000)

In [4]:
def ohlcv_to_df(data, symbol):
    df = pd.DataFrame(data, columns=["timestamp", "open", "high", "low", "close", "volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df[["timestamp", "close"]].copy()
    df = df.rename(columns={"close": symbol})
    return df

eth = ohlcv_to_df(eth_raw, "eth")
sol = ohlcv_to_df(sol_raw, "sol")

In [5]:
df = pd.merge(eth, sol, on="timestamp", how="inner").sort_values("timestamp").reset_index(drop=True)

# Log prices
df["log_eth"] = np.log(df["eth"])
df["log_sol"] = np.log(df["sol"])

df.head()

,timestamp,eth,sol,log_eth,log_sol
0,2026-06-10 00:00:00,1644.64,65.16,7.405277,4.176846
1,2026-06-10 01:00:00,1637.79,65.07,7.401103,4.175464
2,2026-06-10 02:00:00,1623.10,64.29,7.392093,4.163404
3,2026-06-10 03:00:00,1629.41,64.47,7.395973,4.166200
4,2026-06-10 04:00:00,1624.54,64.25,7.392980,4.162782


In [6]:
# 2)
# Create one function that takes two price series and returns:
# hedge ratio β
# spread series
# maybe residuals if you use regression

# A simple first version can be:
# St = log(PA) + βlog(PB)

# This should be shared by both strategies.

In [7]:
import statsmodels.api as sm

In [8]:
X = sm.add_constant(df["log_sol"])
y = df["log_eth"]

ols = sm.OLS(y, X).fit()
print(ols.summary())


                            OLS Regression Results                            
Dep. Variable:                log_eth   R-squared:                       0.265
Model:                            OLS   Adj. R-squared:                  0.264
Method:                 Least Squares   F-statistic:                     258.8
Date:                Thu, 09 Jul 2026   Prob (F-statistic):           5.75e-50
Time:                        17:18:39   Log-Likelihood:                 1330.8
No. Observations:                 721   AIC:                            -2658.
Df Residuals:                     719   BIC:                            -2648.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          6.0339      0.087     69.523      0.0

In [9]:
ols.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                log_eth   R-squared:                       0.265
Model:                            OLS   Adj. R-squared:                  0.264
Method:                 Least Squares   F-statistic:                     258.8
Date:                Thu, 09 Jul 2026   Prob (F-statistic):           5.75e-50
Time:                        17:18:39   Log-Likelihood:                 1330.8
No. Observations:                 721   AIC:                            -2658.
Df Residuals:                     719   BIC:                            -2648.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          6.0339      0.087     69.523      0.000       5.863       6.204
log_sol        0.3254      0.020     16.087      0.000       0.286       0.365
==============================================================================
Omnibus:                       79.899   Durbin-Watson:                   0.013
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               68.413
Skew:                          -0.672   Prob(JB):                     1.39e-15
Kurtosis:                       2.314   Cond. No.                         276.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [10]:
ols.params

const      6.033858
log_sol    0.325423
dtype: float64

In [11]:
ols.resid

0      0.012175
1      0.008451
2      0.003366
3      0.006336
4      0.004455
         ...   
716    0.013920
717    0.014077
718    0.013338
719    0.012305
720    0.012311
Length: 721, dtype: float64

In [13]:
ols.params

const      6.033858
log_sol    0.325423
dtype: float64

In [14]:
Y_est_resid = ols.params['const'] + ols.params['log_sol']*X["log_sol"] + ols.resid
Y_est = ols.params['const'] + ols.params['log_sol']*X["log_sol"]

In [15]:
# 3) Implement the z-score strategy first

# rolling mean of spread
# rolling std of spread
# z-score calculation
# entry rule
# exit rule

# Example:

# enter long spread when z < -2
# enter short spread when z > 2
# exit when z returns near 0


In [16]:
# 5) Spread = regression residual
df["spread"] = ols.resid

# Rolling z-score of spread
window = 25
df["spread_mean"] = df["spread"].rolling(window).mean()
df["spread_std"] = df["spread"].rolling(window).std()
df["zscore"] = (df["spread"] - df["spread_mean"]) / df["spread_std"]

In [17]:
entry_z = 2
exit_z = 0.25

df["signal_spread_z"] = 0
df.loc[df["zscore"] < -entry_z, "signal_spread_z"] = 1
df.loc[df["zscore"] > entry_z, "signal_spread_z"] = -1

In [18]:

# 4) Add the Bollinger Band version

# rolling mean
# rolling std
# upper/lower bands

# Example:

# long when spread crosses below lower band
# short when spread crosses above upper band
# exit at the center line or a fixed threshold


In [19]:
band_k = 1.0

df["upper"] = df["spread_mean"] + band_k * df["spread_std"]
df["lower"] = df["spread_mean"] - band_k * df["spread_std"]

df["signal_spread_bb"] = 0
df.loc[df["spread"] < df["lower"], "signal_spread_bb"] = 1
df.loc[df["spread"] > df["upper"], "signal_spread_bb"] = -1

In [20]:
beta = ols.params["log_sol"]

def make_leg_weights(signal, beta):
    """
    signal =  1 -> long spread
    signal = -1 -> short spread
    signal =  0 -> flat
    """
    w_eth = signal.astype(float)
    w_sol = -beta * signal.astype(float)
    return w_eth, w_sol

df["w_eth_z"], df["w_sol_z"] = make_leg_weights(df["signal_spread_z"], beta)
df["w_eth_bb"], df["w_sol_bb"] = make_leg_weights(df["signal_spread_bb"], beta)

In [21]:
# 5) Write one backtester that both strategies use

# Your backtester should handle:

# entering long/short spread positions
# closing positions
# transaction costs
# PnL calculation
# equity curve
# position sizing

In [22]:
df["ret_eth"] = df["eth"].pct_change()
df["ret_sol"] = df["sol"].pct_change()

In [23]:
df["strategy_ret_z"] = (
    df["w_eth_z"].shift(1) * df["ret_eth"] +
    df["w_sol_z"].shift(1) * df["ret_sol"]
)

df["strategy_ret_bb"] = (
    df["w_eth_bb"].shift(1) * df["ret_eth"] +
    df["w_sol_bb"].shift(1) * df["ret_sol"]
)

In [24]:
# 6) Add evaluation metrics

# Compare strategies using the same metrics:

# total return
# Sharpe ratio
# max drawdown
# win rate
# average trade return
# number of trades
# average holding time
# turnover

# If possible, also compare:

# how often they trade
# how sensitive they are to transaction costs
# how stable results are across different periods


In [25]:
def stats(r):
    r = r.dropna()
    ann_factor = np.sqrt(24 * 365)
    out = {
        "mean_hourly_return": r.mean(),
        "vol_hourly": r.std(),
        "sharpe": (r.mean() / r.std()) * ann_factor if r.std() != 0 else np.nan,
        "cumulative_return": (1 + r).prod() - 1
    }
    return pd.Series(out)

print("Z-score strategy")
print(stats(df["strategy_ret_z"]))

print("\nBollinger strategy")
print(stats(df["strategy_ret_bb"]))

Z-score strategy
mean_hourly_return   -0.000014
vol_hourly            0.001654
sharpe               -0.818259
cumulative_return    -0.011333
dtype: float64

Bollinger strategy
mean_hourly_return    0.000155
vol_hourly            0.003097
sharpe                4.681587
cumulative_return     0.114132
dtype: float64


In [26]:
# 7) Run one pair first, then expand

# Do not start with many pairs.

# Use one pair first, get the code working, then extend to:

# multiple crypto pairs
# different windows
# different thresholds
# out-of-sample testing